In [1]:
%pip install -U langgraph langchain-classic langchain-community langchain-google-genai chromadb


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from typing import List, Optional
from typing_extensions import TypedDict

from langgraph.graph import END, START, StateGraph

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from IPython.display import Markdown, display
import os

from langchain_community.vectorstores import Chroma
from langchain_classic.storage import LocalFileStore
from langchain_classic.retrievers import MultiVectorRetriever
from langchain_google_genai import GoogleGenerativeAIEmbeddings

/var/folders/9x/22rg_ft56flgmbxbwcvxwty80000gn/T/ipykernel_94254/3309195706.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [3]:
from api_config import configure_google_api_key

GOOGLE_API_KEY = configure_google_api_key()

## Retriever

In [4]:
def retrieve_documents(retriever, query):
    """
    Compatible with different LangChain retriever versions.
    """
    if hasattr(retriever, "invoke"):
        return retriever.invoke(query)
    if hasattr(retriever, "get_relevant_documents"):
        return retriever.get_relevant_documents(query)
    raise AttributeError("Le retriever ne supporte ni invoke ni get_relevant_documents.")

In [5]:
CHROMA_DIR = "db/chroma_minecraft_multivec"
STORE_DIR = "db/local_chunks_store"

gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="retrieval_query"
)

vectorstore = Chroma(
    collection_name="minecraft_summaries",
    persist_directory=CHROMA_DIR,
    embedding_function=gemini_embeddings
)

fs_store = LocalFileStore(STORE_DIR)

retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=fs_store,
    id_key="doc_id",
)

print("Total summaries in Chroma:", vectorstore._collection.count())

/var/folders/9x/22rg_ft56flgmbxbwcvxwty80000gn/T/ipykernel_94254/2438023732.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Total summaries in Chroma: 173


#### Initialize Generator

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0
)

# Modèle local Ollama pour les tâches simples (Graders) afin d'éviter l'erreur 429
ollama_llm = ChatOllama(
    model="llama3", # Modifiez ce nom selon le modèle que vous avez installé localement (ex: qwen2.5, mistral)
    temperature=0
)

format_docs:这个函数会把 retrieved chunks 整理成 final generation 和 hallucination grader 能读的 context。

In [7]:
def format_docs(docs):
    """
    Format retrieved documents into a readable context for the LLM.
    """
    formatted_docs = []

    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "unknown")
        section = doc.metadata.get("section", "unknown")

        formatted_text = (
            f"[Document {i}]\n"
            f"Source: {source}\n"
            f"Section: {section}\n"
            f"Content:\n{doc.page_content}"
        )

        formatted_docs.append(formatted_text)

    return "\n\n".join(formatted_docs)

#### Create prompt templates

In [8]:
# Prompt template for final answer generation
llm_prompt_template = """Tu es un assistant expert sur le jeu Minecraft.

Réponds à la question en utilisant UNIQUEMENT le contexte fourni ci-dessous.

Si la réponse ne se trouve pas dans le contexte ou si tu n'es pas sûr, n'invente rien.
Dans ce cas, dis EXACTEMENT :
"Je suis désolé, mais l'information n'est pas dans les documents fournis."

Sois concis, clair, et cite la ou les sources à la fin de ta réponse si tu as trouvé l'information.

Question:
{question}

Contexte:
{context}

Réponse:
"""

llm_prompt = PromptTemplate.from_template(llm_prompt_template)

generation_chain = (
    llm_prompt
    | llm.with_fallbacks([ollama_llm])
    | StrOutputParser()
)

### Graders

#### Retrieval Grader

In [9]:
retrieval_grader_prompt_template = """Tu es un évaluateur de documents pour un système RAG sur Minecraft.

Ton rôle est de dire si le document récupéré est utile pour répondre à la question de l'utilisateur.

Critères :
- Réponds YES si le document contient des informations directement utiles.
- Réponds YES si le document contient des mots-clés ou un sens proche de la question.
- Réponds NO si le document est hors sujet.
- Réponds NO si le document est trop vague pour aider à répondre.

Tu dois répondre uniquement par YES ou NO.

Question utilisateur:
{question}

Document récupéré:
{document}

Réponse:
"""

retrieval_grader_prompt = PromptTemplate.from_template(retrieval_grader_prompt_template)

retrieval_grader = (
    retrieval_grader_prompt
    | ollama_llm
    | StrOutputParser()
)


def grade_one_document(question, document):
    """
    Return True if the document is relevant to the question.
    """
    result = retrieval_grader.invoke({
        "question": question,
        "document": document.page_content[:3000]
    })

    result = result.strip().upper()

    return result.startswith("YES")

#### Hallucination grader
判断最终生成的回答是否真的被 retrieved documents 支撑。

In [10]:
hallucination_grader_prompt_template = """Tu es un évaluateur pour un système RAG.

Ton rôle est de vérifier si la réponse générée est bien soutenue par les documents fournis.

Réponds YES si la réponse est globalement fondée sur les documents.
Réponds NO si la réponse contient des informations inventées ou non soutenues par les documents.

Tu dois répondre uniquement par YES ou NO.

Documents:
{documents}

Réponse générée:
{generation}

Réponse:
"""

hallucination_grader_prompt = PromptTemplate.from_template(hallucination_grader_prompt_template)

hallucination_grader = (
    hallucination_grader_prompt
    | ollama_llm
    | StrOutputParser()
)


def grade_hallucination(documents, generation):
    docs_text = format_docs(documents)

    result = hallucination_grader.invoke({
        "documents": docs_text[:8000],
        "generation": generation
    })

    result = result.strip().upper()

    return result.startswith("YES")

#### Answer Grader


In [11]:
answer_grader_prompt_template = """Tu es un évaluateur pour un système RAG.

Ton rôle est de vérifier si la réponse générée répond réellement à la question utilisateur.

Réponds YES si la réponse répond clairement à la question.
Réponds NO si la réponse est trop vague, hors sujet, ou si elle dit qu'elle ne sait pas.

Tu dois répondre uniquement par YES ou NO.

Question utilisateur:
{question}

Réponse générée:
{generation}

Réponse:
"""

answer_grader_prompt = PromptTemplate.from_template(answer_grader_prompt_template)

answer_grader = (
    answer_grader_prompt
    | ollama_llm
    | StrOutputParser()
)


def grade_answer(question, generation):
    result = answer_grader.invoke({
        "question": question,
        "generation": generation
    })

    result = result.strip().upper()

    return result.startswith("YES")


#### Temporary query rewriter（Multi-Query）

In [12]:
multi_query_prompt_template = """Tu es un assistant expert Minecraft.

Ta tâche est de générer 3 reformulations différentes de la question ci-dessous
afin d'améliorer la recherche dans une base vectorielle.

Chaque reformulation doit aborder la question sous un angle légèrement différent :
synonymes, niveau d'abstraction, point de vue.

Règles :
- français uniquement
- ne réponds pas à la question
- renvoie uniquement les 3 requêtes
- une requête par ligne
- pas de numérotation
- pas de commentaire

Question originale :
{question}

Requêtes :
"""

multi_query_prompt = PromptTemplate.from_template(multi_query_prompt_template)

multi_query_chain = (
    multi_query_prompt
    | llm.with_fallbacks([ollama_llm])
    | StrOutputParser()
)


def generate_multi_queries(question):
    """
    Generate several reformulated queries for Multi-Query retrieval.
    """

    raw_output = multi_query_chain.invoke({"question": question})

    queries = []

    for line in raw_output.strip().split("\n"):
        query = line.strip()

        # Clean possible bullets or numbering, just in case the LLM does not fully follow the prompt.
        query = query.lstrip("-•0123456789. )").strip()

        if query:
            queries.append(query)

    # Keep the original question as the first query.
    all_queries = [question] + queries

    # Simple deduplication.
    unique_queries = []
    seen = set()

    for q in all_queries:
        if q not in seen:
            seen.add(q)
            unique_queries.append(q)

    return unique_queries


def get_unique_documents(lists_of_docs):
    """
    Merge several lists of documents and remove duplicates.
    """

    seen = set()
    unique_docs = []

    for docs in lists_of_docs:
        for doc in docs:
            key = doc.page_content + str(sorted(doc.metadata.items()))

            if key not in seen:
                seen.add(key)
                unique_docs.append(doc)

    return unique_docs


def retrieve_documents_multi_query(retriever, queries):
    """
    Retrieve documents for several queries, then merge and deduplicate results.
    """

    all_results = []

    for i, query in enumerate(queries, start=1):
        print(f"Multi-query {i}: {query}")

        docs = retrieve_documents(retriever, query)
        print(f"Documents retrieved for query {i}: {len(docs)}")

        all_results.append(docs)

    unique_docs = get_unique_documents(all_results)

    print(f"Unique documents after multi-query: {len(unique_docs)}")

    return unique_docs

In [13]:
import re
hyde_dense_template = """Tu es un expert technique du jeu Minecraft.

À partir de la question utilisateur, génère un court document hypothétique qui contiendrait probablement les informations nécessaires pour répondre à cette question.

Ce texte ne doit pas être utilisé comme réponse finale.
Il sert uniquement à améliorer la recherche sémantique dans une base documentaire.

Contraintes :
- français uniquement
- maximum 30 mots
- pas de markdown
- pas d'introduction
- uniquement des faits ou mots-clés pertinents

Question:
{question}

Document hypothétique dense:
"""

hyde_dense_prompt = PromptTemplate.from_template(hyde_dense_template)

hyde_chain = (
    hyde_dense_prompt
    | llm.with_fallbacks([ollama_llm])
    | StrOutputParser()
)


def generate_hyde_query(question):
    hyde_query = hyde_chain.invoke({"question": question})
    hyde_query = hyde_query.strip()
    hyde_query = re.sub(r'[\*`_#]', '', hyde_query)
    return hyde_query

### GraphState

#### Define GraphState

In [14]:
class GraphState(TypedDict):
    """
    State of the Self-RAG graph.

    query_strategy:
        - "standard": original user question
        - "multi_query": several reformulated queries
        - "hyde": hypothetical document generated by HyDE
    """

    question: str
    retrieval_query: str
    documents: List[Document]
    generation: Optional[str]

    query_strategy: str
    multi_queries: Optional[List[str]]
    hyde_query: Optional[str]

    rewrite_count: int
    max_rewrites: int

In [15]:
def get_next_query_translation_route(state):
    """
    Decide the next query translation strategy.

    Priority:
    1. standard query
    2. multi-query
    3. HyDE query
    """

    current_strategy = state.get("query_strategy", "standard")

    if current_strategy == "standard":
        return "transform_query"

    if current_strategy == "multi_query":
        return "transform_query_hyde"

    return None

#### Node 1: retrieve

In [16]:
def retrieve(state):
    """
    Retrieve documents using the current retrieval strategy.
    """

    print("---RETRIEVE---")

    question = state["question"]
    retrieval_query = state["retrieval_query"]
    query_strategy = state.get("query_strategy", "standard")
    multi_queries = state.get("multi_queries")

    print(f"Query strategy: {query_strategy}")
    print(f"Original question: {question}")
    print(f"Retrieval query: {retrieval_query}")

    if query_strategy == "multi_query":
        documents = retrieve_documents_multi_query(
            retriever=retriever,
            queries=multi_queries
        )
    else:
        documents = retrieve_documents(retriever, retrieval_query)

    print(f"Retrieved documents: {len(documents)}")

    return {
        **state,
        "question": question,
        "retrieval_query": retrieval_query,
        "documents": documents,
        "query_strategy": query_strategy,
        "multi_queries": multi_queries,
    }

#### Node 2: grade_documents

In [17]:
def grade_documents(state):
    """
    Grade retrieved documents.
    Keep only relevant documents.
    """

    print("---CHECK DOCUMENT RELEVANCE---")

    question = state["question"]
    retrieval_query = state["retrieval_query"]
    documents = state["documents"]
    query_strategy = state.get("query_strategy", "standard")

    print(f"Query strategy used: {query_strategy}")

    filtered_docs = []

    for i, doc in enumerate(documents, start=1):
        is_relevant = grade_one_document(question, doc)

        source = doc.metadata.get("source", "unknown")

        if is_relevant:
            print(f"---DOCUMENT {i}: RELEVANT | source: {source}---")
            filtered_docs.append(doc)
        else:
            print(f"---DOCUMENT {i}: NOT RELEVANT | source: {source}---")

    print(f"Relevant documents kept: {len(filtered_docs)}")

    return {
        **state,
        "question": question,
        "retrieval_query": retrieval_query,
        "documents": filtered_docs,
        "query_strategy": query_strategy,
    }

#### Node 3: transform_query

In [18]:
def transform_query(state):
    """
    Generate multiple reformulated queries to improve retrieval.
    This replaces the previous rewrite strategy.
    """

    print("---TRANSFORM QUERY WITH MULTI-QUERY---")

    question = state["question"]
    rewrite_count = state.get("rewrite_count", 0)

    multi_queries = generate_multi_queries(question)

    print("Generated multi-queries:")
    for q in multi_queries:
        print(f"- {q}")

    return {
        **state,
        "retrieval_query": " | ".join(multi_queries),
        "multi_queries": multi_queries,
        "documents": [],
        "query_strategy": "multi_query",
        "rewrite_count": rewrite_count + 1,
    }

In [19]:
def transform_query_hyde(state):
    """
    Generate a HyDE query to improve retrieval.
    This is the second fallback query translation strategy.
    """

    print("---TRANSFORM QUERY WITH HYDE---")

    question = state["question"]
    rewrite_count = state.get("rewrite_count", 0)

    hyde_query = generate_hyde_query(question)

    print(f"Original question: {question}")
    print(f"HyDE query: {hyde_query}")

    return {
        **state,
        "retrieval_query": hyde_query,
        "hyde_query": hyde_query,
        "documents": [],
        "query_strategy": "hyde",
        "rewrite_count": rewrite_count + 1,
    }

#### Node 4: generate

In [20]:
def generate(state):
    """
    Generate answer using filtered documents.
    """

    print("---GENERATE---")

    question = state["question"]
    retrieval_query = state["retrieval_query"]
    documents = state["documents"]
    query_strategy = state.get("query_strategy", "standard")

    print(f"Generation based on query strategy: {query_strategy}")

    context = format_docs(documents)

    generation = generation_chain.invoke({
        "question": question,
        "context": context
    })

    return {
        **state,
        "question": question,
        "retrieval_query": retrieval_query,
        "documents": documents,
        "generation": generation,
        "query_strategy": query_strategy,
    }

#### Conditional edge 1: grade_documents

In [21]:
def decide_after_document_grading(state):
    """
    Decide whether to generate an answer or try another query translation strategy.
    """

    print("---ASSESS GRADED DOCUMENTS---")

    filtered_documents = state["documents"]
    query_strategy = state.get("query_strategy", "standard")

    min_relevant_docs = 1

    if len(filtered_documents) >= min_relevant_docs:
        print("---DECISION: DOCUMENTS ARE SUFFICIENT, GENERATE---")
        return "generate"

    next_route = get_next_query_translation_route(state)

    if next_route is not None:
        print(
            f"---DECISION: DOCUMENTS ARE NOT SUFFICIENT "
            f"WITH {query_strategy.upper()}, TRY {next_route.upper()}---"
        )
        return next_route

    print("---DECISION: NO MORE QUERY TRANSLATION STRATEGY, GENERATE WITH EMPTY OR WEAK CONTEXT---")
    return "generate"

#### Conditional edge 2: generation

In [22]:
def grade_generation_v_documents_and_question(state):
    """
    Check if generation is grounded in documents and answers the original question.
    If not, try the next query translation strategy.
    """

    print("---CHECK GENERATION QUALITY---")

    question = state["question"]
    documents = state["documents"]
    generation = state["generation"]
    query_strategy = state.get("query_strategy", "standard")

    # If no documents are available, stop.
    if not documents:
        print("---NO DOCUMENTS AVAILABLE, END---")
        return "end"

    # 1. Hallucination check
    grounded = grade_hallucination(documents, generation)

    if not grounded:
        print("---DECISION: GENERATION IS NOT GROUNDED IN DOCUMENTS---")

        next_route = get_next_query_translation_route(state)

        if next_route is not None:
            print(f"---DECISION: TRY NEXT QUERY TRANSLATION STRATEGY: {next_route.upper()}---")
            return next_route

        print("---DECISION: NO MORE QUERY TRANSLATION STRATEGY, END---")
        return "end"

    print("---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---")

    # 2. Answer quality check
    useful = grade_answer(question, generation)

    if useful:
        print("---DECISION: GENERATION ANSWERS THE QUESTION, END---")
        return "useful"

    print("---DECISION: GENERATION DOES NOT ANSWER THE QUESTION---")

    next_route = get_next_query_translation_route(state)

    if next_route is not None:
        print(f"---DECISION: TRY NEXT QUERY TRANSLATION STRATEGY: {next_route.upper()}---")
        return next_route

    print("---DECISION: NO MORE QUERY TRANSLATION STRATEGY, END---")
    return "end"

## Build LangGraph

In [23]:
workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("transform_query", transform_query)
workflow.add_node("transform_query_hyde", transform_query_hyde)
workflow.add_node("generate", generate)

# Build graph
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "grade_documents")

workflow.add_conditional_edges(
    "grade_documents",
    decide_after_document_grading,
    {
        "transform_query": "transform_query",
        "transform_query_hyde": "transform_query_hyde",
        "generate": "generate",
    },
)

workflow.add_edge("transform_query", "retrieve")
workflow.add_edge("transform_query_hyde", "retrieve")

workflow.add_conditional_edges(
    "generate",
    grade_generation_v_documents_and_question,
    {
        "useful": END,
        "transform_query": "transform_query",
        "transform_query_hyde": "transform_query_hyde",
        "end": END,
    },
)

app = workflow.compile()

In [24]:
def build_initial_state(question: str):
    """
    Build the initial state for the Self-RAG workflow from a simple user question.
    The user only provides the question; all other fields are internal state variables.
    """

    return {
        "question": question,
        "retrieval_query": question,
        "documents": [],
        "generation": None,

        "query_strategy": "standard",
        "multi_queries": None,
        "hyde_query": None,

        "rewrite_count": 0,
        "max_rewrites": 2,
    }

In [25]:
def answer_question(question: str, show_steps: bool = True):
    """
    Run the full Self-RAG workflow from a single user question.
    """

    inputs = build_initial_state(question)

    final_state = None

    if show_steps:
        for output in app.stream(inputs):
            for node_name, state_value in output.items():
                print(f"\nNode finished: {node_name}")
                final_state = state_value
    else:
        final_state = app.invoke(inputs)

    answer = final_state["generation"]

    print("\n" + "=" * 80)
    print("FINAL ANSWER")
    print("=" * 80)
    print(answer)

    print("\n" + "=" * 80)
    print("QUERY TRANSLATION INFO")
    print("=" * 80)
    print("Final query strategy:", final_state.get("query_strategy"))
    print("Multi-queries:", final_state.get("multi_queries"))
    print("HyDE query:", final_state.get("hyde_query"))

    return final_state

## TEST

In [26]:
final_state = answer_question(
    "Quel est l'ingrédient de base indispensable pour l'alchimie ?"
)

---RETRIEVE---
Query strategy: standard
Original question: Quel est l'ingrédient de base indispensable pour l'alchimie ?
Retrieval query: Quel est l'ingrédient de base indispensable pour l'alchimie ?
Retrieved documents: 4

Node finished: retrieve
---CHECK DOCUMENT RELEVANCE---
Query strategy used: standard
---DOCUMENT 1: RELEVANT | source: https://minecraft.fandom.com/fr/wiki/Alchimie---
---DOCUMENT 2: RELEVANT | source: https://minecraft.fandom.com/fr/wiki/Alchimie---
---DOCUMENT 3: NOT RELEVANT | source: https://minecraft.fandom.com/fr/wiki/Alchimie---
---DOCUMENT 4: RELEVANT | source: https://minecraft.fandom.com/fr/wiki/Tutoriels/Survie_dans_le_Nether---
Relevant documents kept: 3
---ASSESS GRADED DOCUMENTS---
---DECISION: DOCUMENTS ARE SUFFICIENT, GENERATE---

Node finished: grade_documents
---GENERATE---
Generation based on query strategy: standard
---CHECK GENERATION QUALITY---
---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---
---DECISION: GENERATION DOES NOT ANSWER THE QUEST

In [27]:
final_state = answer_question(
    "Avec quoi commence-t-on pour préparer une potion dans Minecraft ?",
    show_steps=False
)

---RETRIEVE---
Query strategy: standard
Original question: Avec quoi commence-t-on pour préparer une potion dans Minecraft ?
Retrieval query: Avec quoi commence-t-on pour préparer une potion dans Minecraft ?
Retrieved documents: 4
---CHECK DOCUMENT RELEVANCE---
Query strategy used: standard
---DOCUMENT 1: RELEVANT | source: https://minecraft.fandom.com/fr/wiki/Alchimie---
---DOCUMENT 2: RELEVANT | source: https://minecraft.fandom.com/fr/wiki/Alchimie---
---DOCUMENT 3: RELEVANT | source: https://minecraft.fandom.com/fr/wiki/Alchimie---
---DOCUMENT 4: RELEVANT | source: https://minecraft.fandom.com/fr/wiki/Alchimie---
Relevant documents kept: 4
---ASSESS GRADED DOCUMENTS---
---DECISION: DOCUMENTS ARE SUFFICIENT, GENERATE---
---GENERATE---
Generation based on query strategy: standard
---CHECK GENERATION QUALITY---
---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---
---DECISION: GENERATION ANSWERS THE QUESTION, END---

FINAL ANSWER
Pour préparer une potion dans Minecraft, on commence avec 